# 26 Event-method player-season projection

各 event-level 手法の EV / LA / hard-hit / barrel / xBA などの予測を batter-season 単位へ集約し、BA / OBP / SLG / OPS などの player-season stats と比較できる `predictions_v1` を作ります。fusion 後に再実行すると fusion の player-season projection も作ります。

In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import subprocess
import sys


def _mount_drive_if_needed() -> None:
    try:
        from google.colab import drive  # type: ignore
    except ModuleNotFoundError:
        print('Google Colab ではないため Drive mount を skip します。')
        return

    mountpoint = Path('/content/drive')
    mountpoint.mkdir(parents=True, exist_ok=True)
    if (mountpoint / 'MyDrive').exists():
        print('Drive already mounted at /content/drive')
        return
    try:
        drive.mount(str(mountpoint))
    except ValueError as exc:
        message = str(exc)
        if 'Mountpoint must not already contain files' in message or 'already mounted' in message:
            print(f'Drive mount warning: {message}')
            if not (mountpoint / 'MyDrive').exists():
                drive.mount(str(mountpoint), force_remount=True)
        else:
            raise
    except Exception as exc:
        print(f'Colab Drive mount skipped or failed: {exc}')


os.environ.setdefault('BATTING_CODE_ROOT', '/content/drive/MyDrive/codex/batting_codex_handoff')


def _is_repo_dir(path: Path) -> bool:
    return (
        (path / 'src' / 'sport_pipeline' / '__init__.py').exists()
        and (path / 'configs').exists()
        and (path / 'notebooks').exists()
    )


def _resolve_repo_dir() -> Path:
    fixed = Path(os.environ['BATTING_CODE_ROOT'])
    mydrive = Path('/content/drive/MyDrive')
    candidates = [
        fixed,
        Path('/content/drive/MyDrive/codex/batting_codex_handoff'),
        Path('/content/drive/MyDrive/batting_codex_handoff'),
        Path.cwd(),
        *Path.cwd().parents,
    ]
    checked = []
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if str(candidate) in checked:
            continue
        checked.append(str(candidate))
        if _is_repo_dir(candidate):
            return candidate
    raise FileNotFoundError(
        'sport_pipeline repo が見つかりません。Drive mount と repo 配置を確認してください.\n'
        f'BATTING_CODE_ROOT={fixed}\n'
        '推奨配置: /content/drive/MyDrive/codex/batting_codex_handoff\n'
        '別配置の場合は notebook の最初の cell より前に次を実行してください.\n'
        '  %env BATTING_CODE_ROOT=/content/drive/MyDrive/あなたの配置/batting_codex_handoff\n'
        f'MyDrive exists={mydrive.exists()}\n'
        '確認した候補:\n- ' + '\n- '.join(checked)
    )


_mount_drive_if_needed()
REPO_DIR = _resolve_repo_dir()
os.environ['BATTING_CODE_ROOT'] = str(REPO_DIR)
BASE_DIR = Path('/content/drive/MyDrive/baseball_vision')
CACHE_DIR = Path('/content/cache/baseball_vision')
RUN_PROFILE_NAME = os.environ.get('BASEBALL_VISION_RUN_PROFILE', 'mlb_2024_2026_real_colab_v2.json')
RUN_PROFILE_PATH = REPO_DIR / 'configs' / 'runs' / RUN_PROFILE_NAME
src_dir = REPO_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
if importlib.util.find_spec('sport_pipeline') is None:
    raise ModuleNotFoundError(f'sport_pipeline を import できません: {src_dir}')

from sport_pipeline.pipeline.run_profile import load_run_profile, resolve_statcast_date_range, run_id

RUN_PROFILE = load_run_profile(RUN_PROFILE_PATH)
START_DATE, END_DATE = resolve_statcast_date_range(RUN_PROFILE)
FUSION_RUN_ID = run_id(RUN_PROFILE, 'fusion_run_id')
print('STATCAST_DATE_RANGE =', START_DATE, 'to', END_DATE)
print('RUN_PROFILE_PATH =', RUN_PROFILE_PATH)
print('FUSION_RUN_ID =', FUSION_RUN_ID)


In [ ]:
from sport_pipeline.models.player_season.event_projection import projection_run_id, run_event_prediction_player_season_projection

RUN_CONTEXT_PROJECTION = True
RUN_FUSION_PROJECTION_IF_EXISTS = True
REQUIRE_NON_EMPTY_PROJECTION = False


def prediction_exists(run_id_value: str) -> bool:
    for suffix in ('.parquet', '.jsonl', '.json', '.csv'):
        if (BASE_DIR / 'predictions' / run_id_value / f'predictions_v1{suffix}').exists():
            return True
    return False

candidate_run_keys = [
    'context_run_id',
    'sequence_run_id',
    'sequence_tcn_run_id',
    'video_lightweight_run_id',
    'video_frozen_run_id',
    'video_finetune_run_id',
    'vlm_run_id',
]
source_run_ids = []
for key in candidate_run_keys:
    try:
        rid = run_id(RUN_PROFILE, key)
    except KeyError:
        continue
    if key == 'context_run_id' and not RUN_CONTEXT_PROJECTION:
        continue
    if rid not in source_run_ids:
        source_run_ids.append(rid)
if RUN_FUSION_PROJECTION_IF_EXISTS:
    source_run_ids.append(FUSION_RUN_ID)

completed = []
skipped = []
for index, source_run_id in enumerate(source_run_ids, start=1):
    if source_run_id.endswith('_player_season_projection'):
        continue
    out_run_id = projection_run_id(source_run_id)
    print()
    print('=' * 96)
    print(f'player-season projection {index}/{len(source_run_ids)}: {source_run_id} -> {out_run_id}')
    print('=' * 96)
    if not prediction_exists(source_run_id):
        print('SKIP missing source predictions:', source_run_id)
        skipped.append({'source_run_id': source_run_id, 'reason': 'missing_source_predictions'})
        continue
    outputs = run_event_prediction_player_season_projection(
        BASE_DIR,
        source_run_id=source_run_id,
        output_run_id=out_run_id,
        require_non_empty=REQUIRE_NON_EMPTY_PROJECTION,
    )
    completed.append({'source_run_id': source_run_id, 'output_run_id': out_run_id, **{k: str(v) for k, v in outputs.items()}})
    for name, path in outputs.items():
        print(name, '->', path, 'exists=', path.exists())

summary_path = BASE_DIR / 'reports/preflight/event_method_player_season_projection_summary_v1.json'
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary_path.write_text(json.dumps({'completed': completed, 'skipped': skipped}, ensure_ascii=False, indent=2), encoding='utf-8')
print('summary_path =', summary_path)
